# CredResolve Collections Analytics — Investigation Notebook

## Objective
Test the claim that collections recovery improved by 11% month-on-month, reconstruct a transparent recovery metric, audit data-quality issues, and distinguish descriptive evidence from causal claims.

**Scope caveat:** the supplied extract covers January–8 August 2026; August is partial. It does not contain a full 12-month period.

## Metric definition used here
- Deduplicate payment events by `payment_id`, keeping the first source row for reproducibility.
- Count only `payment_status == SUCCESS` as recovered amount; pending, failed, and reversed events are excluded from recovery.
- Convert event timestamps from the payment provider's declared timezone to Asia/Kolkata before assigning the month.
- Normalize monthly recovery by observed calendar days to avoid comparing a partial month or unequal month lengths directly.
- Use the full account book as the denominator for book-level rates; do not filter accounts based on contact or payment outcomes.

This is a defensible independent definition, but final production reporting should reconcile successful events to settlement/reversal records and confirm provider timezone metadata.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src" / "run_analysis.py").exists():
    ROOT = ROOT / "credresolve-assignment"
DATA = ROOT / "data"
REPORTS = ROOT / "reports"
print("Project:", ROOT.resolve())
print("CSV files found:", len(list(DATA.glob("*.csv"))))

## 1. Reproduce the analysis

The analysis script reads raw CSVs without modifying them and writes the monthly metrics, audit tables, charts, and one-screen HTML dashboard. It can be rerun from the project root after copying the dataset into `data/`.

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "src/run_analysis.py", "--data", "data", "--out", "reports"],
    cwd=ROOT, text=True, capture_output=True
)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError("Analysis script failed; inspect the error above.")

## 2. What happened? Monthly recovery

Monthly raw totals are not comparable when month lengths differ, and August is only eight observed days. Review both total recovered amount and recovery per observed day.

In [ ]:
monthly = pd.read_csv(REPORTS / "monthly_recovery.csv")
display(monthly[["month", "recovered_amount", "days_observed", "recovered_per_day",
                 "mom_per_day_pct", "payers", "recovery_rate_30d_pct"]].round(3))
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly["month"], monthly["recovered_per_day"] / 1e6, marker="o")
ax.set_title("Recovery per observed day (IST-normalized)")
ax.set_ylabel("₹ million per day")
ax.tick_params(axis="x", rotation=45)
ax.grid(True, alpha=.25)
plt.tight_layout()

### Interpretation
January-to-July recovery per day changes by approximately **+0.09%** under the stated rules. February-to-March deduplicated total recovery rises **11.46%**, but the daily-normalized change is only **+0.68%** because March has 31 days versus February's 28. Monthly movement is mixed, so this does not establish sustained 11% operational improvement. The original company metric definition and full 12-month report are needed to verify its exact claim.

## 3. Data forensics

The next table quantifies duplicate IDs and distinguishes a repeated amount from genuine additional cash. Rejected rows should remain available in quarantine tables with reason codes and lineage.

In [ ]:
dq = pd.read_csv(REPORTS / "data_quality_summary.csv")
display(dq)
tz = pd.read_csv(REPORTS / "timezone_audit.csv")
display(tz)

### Interpretation
- Duplicate payment IDs are a direct risk of overstating recovery; later duplicate successful rows contain about **₹2.59 Cr** of repeated amount.
- Calls and WhatsApp events also contain duplicate IDs and need deterministic deduplication.
- Timestamp normalization changes the hour classification for a large share of calls. Hour-of-day and contact-window analysis should use normalized local time.
- Versioned borrower and agent records require latest-valid-record logic and preserved history. Matching agent identities on name alone is unsafe.

## 4. Portfolio and targeting mix

Use the monthly segment tables to compare payment incidence across loan types and DPD buckets. These are descriptive comparisons using the current account snapshot, not historical DPD at the time of each payment.

In [ ]:
loan = pd.read_csv(REPORTS / "loan_type_monthly.csv")
display(loan[loan["month"].isin(["2026-01", "2026-07"])].pivot(index="loan_type", columns="month", values="payer_rate_pct").round(2))
dpd = pd.read_csv(REPORTS / "dpd_monthly.csv")
display(dpd[dpd["month"].isin(["2026-01", "2026-07"])].pivot(index="dpd_bucket", columns="month", values="payer_rate_pct").round(2))
mix = pd.read_csv(REPORTS / "targeting_strategy_mix.csv")
display(mix.pivot(index="month", columns="strategy_version", values="target_rows").fillna(0).astype(int))

## 5. Attribution-window sensitivity

A targeting record is not equivalent to a delivered or answered contact, and a last-touch label does not prove that the touch caused payment. The audit compares naive latest targeting with a 30-day prior-target window to expose sensitivity to attribution choices.

In [ ]:
attrib = pd.read_csv(REPORTS / "attribution_audit.csv")
display(attrib.T)

## 6. Selection bias, denominator effects, and cohort outcomes

The same-month targeted/contacted comparison below is a diagnostic only. A payment can occur before the target event, and account status can change multiple times within a month. It must not be presented as a causal channel conversion rate.

In [ ]:
denom = pd.read_csv(REPORTS / "targeting_denominator_audit.csv")
display(denom)
cohort = pd.read_csv(REPORTS / "first_target_cohort_30d.csv")
display(cohort)

### Interpretation
The targeting denominator table exposes how the eligible population changes when the denominator is restricted to accounts marked `CONTACTED`. This is selection bias: the contacted group is not randomly selected, and the status itself may be assigned after an event. The first-target cohort uses a fixed 30-day outcome window and excludes cohorts without full follow-up. Cohort differences are descriptive and may reflect changing portfolio, campaign, DPD, client, geography, and seasonality—not targeting's causal effect.

## 8. Survivorship and denominator manipulation

A metric restricted to currently active accounts can differ from a metric using the fixed book. The current account status is only a snapshot and must not be treated as historical status; this comparison is a sensitivity check, not a preferred denominator.

In [ ]:
status_audit = pd.read_csv(REPORTS / "survivorship_status_audit.csv")
display(status_audit[status_audit["month"].isin(["2026-01", "2026-07"])].pivot(
    index="status", columns="month", values="payer_rate_pct"
).round(3))
display(status_audit[status_audit["month"].isin(["2026-01", "2026-07"])])

### Interpretation
If a reported rate silently excludes closed, paid, written-off, skipped, expired, or otherwise unsuccessful records, the denominator changes and can make performance look better or worse. Use a fixed, predeclared eligible-book denominator and report any exclusions explicitly. Since this file contains a current status snapshot, do not use it to reconstruct historical eligibility without status effective dates.

## 7. Simpson's paradox and mix effects

Compare the fixed-book payer rate with loan-type-specific rates between January and July. Because the denominator is the full account book, loan-type population sizes are fixed in this snapshot; this check can detect differing segment movements but cannot fully reconstruct historical portfolio mix. DPD is also a current snapshot, not historical DPD.

In [ ]:
loan = pd.read_csv(REPORTS / "loan_type_monthly.csv")
jan_jul = loan[loan["month"].isin(["2026-01", "2026-07"])].pivot(index="loan_type", columns="month", values="payer_rate_pct")
jan_jul["change_pp"] = jan_jul["2026-07"] - jan_jul["2026-01"]
display(jan_jul.round(3))
overall = monthly[monthly["month"].isin(["2026-01", "2026-07"])].copy()
overall["payer_rate_pct"] = overall["payers"] / 30000 * 100
display(overall[["month", "payers", "payer_rate_pct"]].round(3))
dpd = pd.read_csv(REPORTS / "dpd_monthly.csv")
display(dpd[dpd["month"].isin(["2026-01", "2026-07"])].pivot(index="dpd_bucket", columns="month", values="payer_rate_pct").round(3))

### Interpretation
The overall payer rate declines slightly from January to July while segment directions differ; this is a reason to inspect mix and segment rates, not proof of a classic Simpson's paradox. A full test requires time-valid account attributes and standardized segment weights. Current attributes can misclassify historical DPD/risk and therefore limit the strength of this conclusion.

## 6. Counterfactual design and investment decision

**Recommended design:** a randomized, stratified holdout pilot for one investment candidate rather than committing the entire ₹10 Cr from observational correlations.

- **Treatment:** eligible accounts randomly assigned to the proposed intervention.
- **Control:** comparable eligible accounts continue current treatment.
- **Strata:** DPD, loan type, client, geography, and channel/operating unit where feasible.
- **Primary outcome:** incremental successful recovery net of reversals and incremental operating cost per eligible account over a pre-registered window.
- **Guardrails:** complaints, repeat contact burden, opt-outs, and borrower outcomes.
- **Analysis:** intention-to-treat difference in means with confidence interval; report downside case and cost per incremental rupee recovered.
- **Assumptions/limits:** no material spillovers, assignment integrity, consistent outcome definitions, adequate sample size, and complete outcome capture.

Break-even occurs when incremental net recovery equals the fully loaded pilot and scale cost. Scale only if the conservative confidence bound on incremental ROI is positive and guardrails remain acceptable.

## 10. Production analytics design

`Raw → Staging → Clean → Golden → Feature → Metrics → Dashboard`

Production controls should include data contracts, stable primary keys, lineage/source-row tracking, deterministic deduplication, timezone normalization, late-arriving-event watermarks, replayable backfills, metric versioning, daily data-quality checks, anomaly alerts, and monitoring of freshness/completeness. See `reports/architecture.mmd` and `reports/executive_memo.md`.

## Final limitations
1. Supplied data does not cover 12 complete months.
2. `SUCCESS` is used as a proxy for recovered cash until settlement/reversal reconciliation is confirmed.
3. Current DPD/account attributes may not represent the state at event time.
4. Targeted accounts are not necessarily reached accounts.
5. Observational comparisons do not identify causal channel lift or investment ROI.